In [ ]:
# Desinstalar la versión conflictiva e instalar el binario probado
!pip install --quiet scikit-learn==1.5.2 scikeras==0.13.0

In [ ]:
# Ejecuta esto UNA SOLA VEZ para reparar el archivo
with open('Churn_Modelling.csv', 'r', encoding='utf-8') as f:
    contenido = f.read()

# Reemplazamos los punto y coma por comas
contenido_arreglado = contenido.replace(';', ',')

with open('Churn_Modelling.csv', 'w', encoding='utf-8') as f:
    f.write(contenido_arreglado)

print("¡Archivo convertido a formato con comas exitosamente!")

¡Archivo convertido a formato con comas exitosamente!


In [ ]:
# Activa el visor interactivo de Colab
%load_ext google.colab.data_table
dataset

The google.colab.data_table extension is already loaded. To reload it, use:
  %reload_ext google.colab.data_table


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [ ]:
# Importar las librerias
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Importar el data set
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, 3:10].values
y = dataset.iloc[:, 10].values

In [ ]:
# Codificar datos categóricos
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# 1. Convertir Género a 0 y 1 (columna 2: 'Female'/'Male')
labelencoder_X_2 = LabelEncoder()
X[:, 2] = labelencoder_X_2.fit_transform(X[:, 2])

# 2. Convertir País a variables Dummy / One-Hot (columna 1: 'France', 'Germany', 'Spain')
ct = ColumnTransformer(
    transformers=[('one_hot_encoder', OneHotEncoder(), [1])],
    remainder='passthrough'
)
X = ct.fit_transform(X)

# 3. Evitar la trampa de la variable ficticia (eliminar la primera columna dummy generada)
X = X[:, 1:]

In [ ]:
# Dividir el data set en conjunto de entrenamiento y conjunto de testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 0)

In [ ]:
# Comando mágico para ver el explorador de variables en la consola
%whos

In [ ]:
# Escalado de variables
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

PARTE 2 - CONSTRUIR LA RNA

In [ ]:
# ARREGLO ERROR

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Cargar datos desde cero
dataset = pd.read_csv('Churn_Modelling.csv')
X = dataset.iloc[:, :-1].values
y = dataset.iloc[:, -1].values

# 2. Codificar género (columna 2)
le = LabelEncoder()
X[:, 2] = le.fit_transform(X[:, 2])

# 3. Codificar geografía (columna 1)
ct = ColumnTransformer(
    transformers=[('one_hot_encoder', OneHotEncoder(), [1])],
    remainder='passthrough'
)
X = ct.fit_transform(X)

# 4. Eliminar primera columna dummy para evitar multicolinealidad
X = X[:, 1:]

# 5. División train / test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

# 6. Escalado de variables
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

# Verificar forma final (debe decir exactamente (..., 11))
print("Forma de X_train:", X_train.shape)

Forma de X_train: (8000, 11)


In [ ]:
# Importar Keras y librerias adicionales

import keras
from keras.models import Sequential
from keras.layers import Dense


In [ ]:
# Incializar la RNA

classifier = Sequential()

In [ ]:
# Añadir las capas de entrada y primera capa oculta

classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu', input_dim = 11))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
# Añadir segunda capa oculta

classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu'))

In [ ]:
# Añadir la capa de salida

classifier.add(Dense(units = 1, kernel_initializer = 'uniform',
                     activation = 'sigmoid'))

In [ ]:
# Compilar la RNA

classifier.compile(optimizer = 'adam',
                   loss = 'binary_crossentropy',
                   metrics = ['accuracy'])

In [ ]:
# Ajustar la RNA al conjunto de entrenamiento

classifier.fit(X_train, y_train, batch_size = 10, epochs = 100)

PARTE 3 - EVALUAR EL MODELO Y CALCULAR PREDICCIONES FINALES

In [ ]:
# Predicción de los resultados con el conjunto de testing
y_pred = classifier.predict(X_test)
y_pred = (y_pred>0.5)

NotFittedError: Estimator needs to be fit before `predict` can be called

In [ ]:
# Elaborar una matriz de confusión
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

In [ ]:
# Nuevo cliente para saber si deja o no el banco

import numpy as np

# Definir el vector con las 11 características en el orden del modelo
cliente_evaluar = np.array([[0, 0, 600, 1, 40, 3, 60000, 2, 1, 1, 50000]])

# Escalar los valores usando la transformación aprendida por StandardScaler
cliente_escalado = sc.transform(cliente_evaluar)

# Predecir la probabilidad de salida
probabilidad_salida = classifier.predict(cliente_escalado)
se_va = probabilidad_salida > 0.5

print(f"Probabilidad de salida calculada: {probabilidad_salida[0][0] * 100:.2f}%")
print(f"¿Deberíamos decirle adiós?: {'SÍ' if se_va[0][0] else 'NO'}")

PARTE 4 - EVALUAR, MEJORAR Y AJUSTAR LA RNA

EVALUAR

In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import cross_val_score
from keras.models import Sequential
from keras.layers import Dense


In [ ]:
def build_classifier():

  # Incializar la RNA
  classifier = Sequential()

  # Añadir las capas de entrada y primera capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu', input_dim = 11))

  # Añadir segunda capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu'))

  # Añadir la capa de salida
  classifier.add(Dense(units = 1, kernel_initializer = 'uniform',
                     activation = 'sigmoid'))

  # Compilar la RNA
  classifier.compile(optimizer = 'adam',
                   loss = 'binary_crossentropy',
                   metrics = ['accuracy'])

  # Devolver el clasificador
  return classifier

In [ ]:
classifier = KerasClassifier(model = build_classifier, batch_size = 10, epochs = 100)
accuracies = cross_val_score(estimator = classifier, X = X_train, y = y_train,
                             cv = 10, n_jobs = -1, verbose = 1)

In [ ]:
mean = accuracies.mean()
variance = accuracies.std()
print(mean)
print(variance)

0.8396250000000001
0.01632722036967713


MEJORAR

Regularización de Dropout para evitar el overfitting

In [ ]:
import keras
from keras.models import Sequential
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Dropout

In [ ]:
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout

# Inicializar la Red Neuronal Artificial
classifier = Sequential()

# 1. Capa de entrada formal (Elimina el UserWarning)
# Se pasa como una tupla indicando el número de variables independientes
classifier.add(Input(shape=(11,)))

# 2. Primera capa oculta
classifier.add(Dense(units=6, kernel_initializer='he_uniform', activation='relu'))
classifier.add(Dropout(rate=0.1))

# 3. Segunda capa oculta
classifier.add(Dense(units=6, kernel_initializer='he_uniform', activation='relu'))
classifier.add(Dropout(rate=0.1))

# 4. Capa de salida (1 neurona para clasificación binaria)
classifier.add(Dense(units=1, kernel_initializer='glorot_uniform', activation='sigmoid'))

# 5. Compilación del grafo computacional
classifier.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

# Visualizar el resumen de arquitectura y dimensiones
classifier.summary()

Model: "sequential_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_20 (Dense)                │ (None, 6)              │            72 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 6)              │            42 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 6)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 1)              │             7 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 121 (484.00 B)

 Trainable params: 121 (484.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
  # Incializar la RNA
  classifier = Sequential()

  # Añadir las capas de entrada y primera capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu', input_dim = 11))

  # No pasar del 0.5 para no tener un desajuste underfitting (modelo demasiado simple)
  classifier.add(Dropout(rate = 0.1))

  # Añadir segunda capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu'))
  classifier.add(Dropout(rate = 0.1))

  # Añadir la capa de salida
  classifier.add(Dense(units = 1, kernel_initializer = 'uniform',
                     activation = 'sigmoid'))

  # Compilar la RNA
  classifier.compile(optimizer = 'adam',
                   loss = 'binary_crossentropy',
                   metrics = ['accuracy'])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


AJUSTAR

In [ ]:
!pip install scikeras

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
from scikeras.wrappers import KerasClassifier

classifier = KerasClassifier(model = build_classifier)

In [ ]:
def build_classifier(optimizer):

  # Incializar la RNA
  classifier = Sequential()

  # Añadir las capas de entrada y primera capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu', input_dim = 11))

  # Añadir segunda capa oculta
  classifier.add(Dense(units = 6, kernel_initializer = 'uniform',
                     activation = 'relu'))

  # Añadir la capa de salida
  classifier.add(Dense(units = 1, kernel_initializer = 'uniform',
                     activation = 'sigmoid'))

  # Compilar la RNA
  classifier.compile(optimizer = optimizer,
                   loss = 'binary_crossentropy',
                   metrics = ['accuracy'])

  # Devolver el clasificador
  return classifier

In [ ]:
classifier = KerasClassifier(model = build_classifier)

In [ ]:
parameters = {
    'batch_size': [25, 32],
    'epochs': [100, 500],
    'optimizer': ['adam', 'rmsprop']
}

In [ ]:
import sklearn.base

# Parche definitivo para el bug de __sklearn_tags__ en scikit-learn 1.6+
def _patched_classifier_tags(self):
    tags = sklearn.base.BaseEstimator.__sklearn_tags__(self)
    tags.estimator_type = "classifier"
    try:
        from sklearn.utils._tags import ClassifierTags
        tags.classifier_tags = ClassifierTags()
    except Exception:
        pass
    return tags

sklearn.base.ClassifierMixin.__sklearn_tags__ = _patched_classifier_tags

In [ ]:
grid_search = GridSearchCV(estimator = classifier,
                          param_grid = parameters,
                          scoring = 'accuracy',
                          cv = 10)
grid_search = grid_search.fit(X_train, y_train)
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_




In [ ]:
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
from keras.models import Sequential
from keras.layers import Input, Dense, Dropout

# 1. Función constructora con valor por defecto para el argumento 'optimizer'
# Esto evita que falle si se llama sin parámetros
def build_classifier(optimizer='adam'):
    classifier = Sequential()

    # Capa de entrada formal
    classifier.add(Input(shape=(11,)))

    # Capas ocultas con Dropout
    classifier.add(Dense(units=6, kernel_initializer='he_uniform', activation='relu'))
    classifier.add(Dropout(0.1))

    classifier.add(Dense(units=6, kernel_initializer='he_uniform', activation='relu'))
    classifier.add(Dropout(0.1))

    # Capa de salida binaria
    classifier.add(Dense(units=1, kernel_initializer='glorot_uniform', activation='sigmoid'))

    # Compilación usando la variable 'optimizer'
    classifier.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return classifier

# 2. Instancia del estimador
# verbose=0 evita imprimir 100 épocas de texto en cada uno de los 80 fits
classifier = KerasClassifier(model=build_classifier, verbose=0)

# 3. Grilla de parámetros con sintaxis Scikeras
# REGLA: Los parámetros de la función constructora llevan el prefijo 'model__'
parameters = {
    'batch_size': [25, 32],
    'epochs': [50, 100],
    'model__optimizer': ['adam', 'rmsprop']
}

# 4. Configurar el Grid Search
# n_jobs=1 es fundamental en Google Colab para evitar colisiones de hilos de TensorFlow
grid_search = GridSearchCV(
    estimator=classifier,
    param_grid=parameters,
    scoring='accuracy',
    cv=10,
    n_jobs=1
)

print("Iniciando optimización de hiperparámetros (80 entrenamientos)...")
grid_search = grid_search.fit(X_train, y_train)

# 5. Extracción de los mejores resultados
best_parameters = grid_search.best_params_
best_accuracy = grid_search.best_score_

print("\n--- RESULTADOS OBTENIDOS ---")
print("Mejor combinación encontrada:", best_parameters)
print(f"Mejor precisión media (CV 10-fold): {best_accuracy * 100:.2f}%")

Iniciando optimización de hiperparámetros (80 entrenamientos)...


KeyboardInterrupt: 